# FREUID baseline — full-data training (VESSL Cloud GPU)

Colab 버전과 달리 **ngrok 터널 / Drive 마운트가 필요 없다.**
VESSL Workspace가 파일과 환경을 유지해주기 때문.

## 최초 1회 세팅 순서
1. cloud.vessl.ai → New Workspace → GPU 선택 (A100 권장)
2. Image: `vessl/ml:latest` 또는 `pytorch/pytorch:2.x-cuda12.x-cudnn9-devel`
3. JupyterLab 또는 VS Code SSH로 접속
4. 아래 STEP 1 셀들 순서대로 실행

> **Pause & Resume**: VESSL은 Workspace를 일시정지해도 파일이 유지된다.
> `checkpoints/`, `submissions/` 폴더가 그대로 남으므로 Drive 마운트 불필요.

## STEP 1 — 환경 세팅 (최초 1회)

In [ ]:
# GPU 확인
import torch, os
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu  <- GPU 설정 확인 필요')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('cwd:', os.getcwd())

In [ ]:
# 레포 클론 + 패키지 설치 (최초 1회 — 이미 클론돼 있으면 스킵)
import os
if not os.path.exists('src/freuid'):
    !git clone --depth 1 https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git .
!pip install -q -e .

In [ ]:
# Kaggle 인증
# 권장: VESSL Dashboard -> Workspace -> Environment Variables 에서
#   KAGGLE_USERNAME=xxx  KAGGLE_KEY=yyy 로 주입
import os, json

username = os.environ.get('KAGGLE_USERNAME', '')
key      = os.environ.get('KAGGLE_KEY', '')

if not username or not key:
    # 환경변수 없으면 직접 입력 (실행 후 값 지울 것)
    username = 'YOUR_KAGGLE_USERNAME'
    key      = 'YOUR_KAGGLE_KEY'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('kaggle.json 완료')

In [ ]:
# 데이터 다운로드 (최초 1회, ~수 GB)
# VESSL Workspace는 스토리지가 유지되므로 한 번만 받으면 됨
import os
if not os.path.exists('data/train'):
    !kaggle competitions download -c the-freuid-challenge-2026-ijcai-ecai -p data
    !cd data && unzip -q -o '*.zip' && rm -f *.zip
    print('다운로드 완료')
else:
    print('데이터 이미 존재 — 스킵')

!echo 'train imgs:' && ls data/train/train | wc -l

In [16]:
%%writefile configs/a100_vit_all_strategies.yaml
name: a100_vit_all_strategies
seed: 42
data_dir: data
image_size: 384
val_fraction: 0.1
backbone: vit_base_patch16_384
pretrained: true
epochs: 20
batch_size: 32
lr: 0.0003
weight_decay: 0.0001
num_workers: 4

Writing configs/a100_vit_all_strategies.yaml


In [21]:
CONFIG = 'configs/a100_vit_all_strategies.yaml'
!python -m freuid.train --config {CONFIG}

[train] config 'a100_vit_all_strategies' | device=cuda | backbone=vit_base_patch16_384 | image_size=384 mean=(0.5, 0.5, 0.5)
[train] train=62417 val=6935
Traceback (most recent call last):    
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/kaggle_freuid_2026/src/freuid/train.py", line 148, in <module>
    main()
  File "/content/kaggle_freuid_2026/src/freuid/train.py", line 129, in main
    train_loss, *_ = run_epoch(model, train_loader, device, criterion, optimizer)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/kaggle_freuid_2026/src/freuid/train.py", line 52, in run_epoch
    total_loss += loss.item() * bs
                  ^^^^^^^^^^^
KeyboardInterrupt


In [18]:
CKPT = 'checkpoints/a100_vit_all_strategies.pt'
OUT = 'submissions/a100_vit_all_strategies.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT}
!head -3 {OUT} && wc -l {OUT}

[infer] backbone=vit_base_patch16_384 image_size=384 (from checkpoint) | device=cuda | data_dir=data
[infer] 142818 ids total; 7821 images present locally
[infer] wrote 142818 rows -> submissions/a100_vit_all_strategies.csv (7821 scored, 134997 defaulted)
id,label
0009cd06cf8c426789aa58b2b05c60d2,0.00015405532030854374
000c9f83f98c4babac3f030df5300a8e,0.016383664682507515
142819 submissions/a100_vit_all_strategies.csv


In [19]:
OUT = 'submissions/a100_vit_all_strategies.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'vit all strategies'

100% 5.16M/5.16M [00:02<00:00, 2.19MB/s]
Successfully submitted to The FREUID Challenge 2026 - IJCAI-ECAI

In [2]:
CONFIG = 'configs/dylan_simple_resnet18.yaml'
!python -m freuid.train --config {CONFIG}

[train] config 'dylan_simple_resnet18' | device=cuda | backbone=resnet18 | image_size=224 mean=(0.485, 0.456, 0.406)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[train] train=62417 val=6935
model.safetensors: 100% 46.8M/46.8M [00:01<00:00, 32.3MB/s]
  0% 0/1950 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware tha

: 

---
## DCT Dual-Stream 실험 (RGB + 주파수 branch)

기존 ViT-only 모델에 **DCT branch**를 추가한 듀얼 스트림 버전.
GenAI edits / Print-and-Capture의 JPEG 압축 아티팩트를 주파수 도메인에서 직접 포착한다.

```
이미지
  ├─ RGB stream  → ViT-B/16        → [B, 768]
  └─ DCT stream  → 8x8 BlockDCT   → 경량 CNN → [B, 256]
                    concat [B, 1024] → head → fraud logit
```

> config 파일: `configs/a100_vit_dct.yaml` (이미 repo에 있음 — 아래 셀은 확인용)

In [ ]:
# DCT config 확인 (이미 파일로 존재 — 덮어쓰지 않으려면 이 셀은 스킵)
!cat configs/a100_vit_dct.yaml

In [ ]:
# DCT 듀얼 스트림 학습
# 로그에 'model=DualStream(RGB+DCT)' 뜨면 정상
CONFIG = 'configs/a100_vit_dct.yaml'
!python -m freuid.train --config {CONFIG}

In [ ]:
# DCT 모델 inference + 제출
CKPT = 'checkpoints/a100_vit_dct.pt'
OUT  = 'submissions/a100_vit_dct.csv'
!python -m freuid.infer --checkpoint {CKPT} --out {OUT} --tta 5
!head -3 {OUT} && wc -l {OUT}
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'vit+dct dual stream cross_domain'

## 참고 — Colab과의 차이점

| | Colab | VESSL |
|---|---|---|
| 터널 | ngrok 필요 | SSH/JupyterLab 직접 제공 |
| 파일 유지 | Drive 마운트 필요 | Workspace 자동 유지 |
| 세션 제한 | 90분 idle / 12h max | 크레딧 소진 전까지 유지 |
| Kaggle 인증 | Colab Secrets | 환경변수 또는 직접 입력 |
| Resume 학습 | Drive 통해 체크포인트 | 로컬에 그대로 존재 |

**VESSL에서 학습 중단 후 재개:**
```bash
# _resume.pt 가 자동으로 감지되어 이어서 학습됨
python -m freuid.train --config configs/a100_vit_all_strategies.yaml
```